# 01 - Deploy BYOM Resource Foundation

Run setup first and select **Python (byom01-capability-testing)**.

This adapts the evalgw02 deployment journey to provision APIM **Standard**, Foundry **S0** with a project, and Foundry IQ (Azure AI Search) **S1**.

**Charges continue while APIM and Search exist, even when idle.** The services are not connected; no models, gateway routes, indexes, or evaluations are created.

Bicep defines the infrastructure. Python SDK calls deploy the compiled template and save identifiers/endpoints to `env/.env`. No keys are retrieved.

## Configuration

In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

DEMO_ID = "byom01"
RESOURCE_GROUP = "rg-byom01-uks"
LOCATION = "uksouth"
DEPLOYMENT_NAME = "main-byom01"

demo_root = Path("..").resolve()
bicep_file = demo_root / "infra" / "main.bicep"
env_path = demo_root / "env" / ".env"
if str(demo_root) not in sys.path:
    sys.path.insert(0, str(demo_root))

from app.notebook_helpers import resolve_az_cli

az_cmd = resolve_az_cli()
if not az_cmd:
    raise RuntimeError("Azure CLI not found. Run the setup notebook first.")

print(f"Resource group: {RESOURCE_GROUP}")
print(f"Location: {LOCATION}")
print("Paid tiers: APIM Standard (1 unit), Foundry S0, Search S1 (1 replica, 1 partition)")

## Resolve Subscription and Authentication

Use `AZURE_SUBSCRIPTION_ID` when set; otherwise use the Azure CLI subscription selected during setup. No subscription is selected from a list.

In [ ]:
from azure.identity import AzureCliCredential
from azure.mgmt.resource import ResourceManagementClient

account_cmd = [az_cmd, "account", "show", "--output", "json"]
requested_subscription = os.getenv("AZURE_SUBSCRIPTION_ID")
if requested_subscription:
    account_cmd.extend(["--subscription", requested_subscription])

account_result = subprocess.run(account_cmd, capture_output=True, text=True)
if account_result.returncode != 0:
    raise RuntimeError(
        f"az account show failed (exit {account_result.returncode})\n"
        f"stderr: {account_result.stderr.strip()}\n"
        f"stdout: {account_result.stdout.strip()}"
    )

account = json.loads(account_result.stdout)
subscription_id = account["id"]
credential = AzureCliCredential(tenant_id=account["tenantId"])
resource_client = ResourceManagementClient(credential, subscription_id)
print(f"Subscription: {account['name']} ({subscription_id})")

## Compile Bicep

In [ ]:
build_result = subprocess.run(
    [az_cmd, "bicep", "build", "--file", str(bicep_file), "--stdout"],
    capture_output=True,
    text=True,
)
if build_result.returncode != 0:
    raise RuntimeError(
        f"az bicep build failed (exit {build_result.returncode})\n"
        f"stderr: {build_result.stderr.strip()}\n"
        f"stdout: {build_result.stdout.strip()}"
    )

template = json.loads(build_result.stdout)
print("Bicep compiled")

## Create Resource Group

In [ ]:
resource_client.resource_groups.create_or_update(
    RESOURCE_GROUP,
    {"location": LOCATION},
)
print(f"Created or updated resource group: {RESOURCE_GROUP}")

## Deploy Bicep

Wait for completion; APIM provisioning can take a while. If deployment fails, follow the README's partial-deployment cleanup instructions. Do not run the environment-write step after a failure.

In [ ]:
outputs = None
print("Deploying infrastructure...")
deployment = resource_client.deployments.begin_create_or_update(
    RESOURCE_GROUP,
    DEPLOYMENT_NAME,
    {
        "properties": {
            "mode": "Incremental",
            "template": template,
            "parameters": {
                "demoId": {"value": DEMO_ID},
                "location": {"value": LOCATION},
            },
        }
    },
).result()
outputs = deployment.properties.outputs
print("Deployment completed")
print(json.dumps(outputs, indent=2))

## Write env/.env

In [ ]:
env_values = {
    "AZURE_SUBSCRIPTION_ID": outputs["subscriptionId"]["value"],
    "AZURE_RESOURCE_GROUP": outputs["resourceGroupName"]["value"],
    "AZURE_LOCATION": outputs["location"]["value"],
    "AZURE_ACCOUNT_NAME": outputs["foundryAccountName"]["value"],
    "AZURE_AI_RESOURCE_ID": outputs["foundryResourceId"]["value"],
    "AZURE_AI_ENDPOINT": outputs["foundryEndpoint"]["value"],
    "AZURE_AI_PROJECT_NAME": outputs["projectName"]["value"],
    "AZURE_AI_PROJECT_ID": outputs["projectResourceId"]["value"],
    "AZURE_AI_PROJECT_ENDPOINT": outputs["projectEndpoint"]["value"],
    "AZURE_APIM_NAME": outputs["apimServiceName"]["value"],
    "AZURE_APIM_RESOURCE_ID": outputs["apimResourceId"]["value"],
    "APIM_ENDPOINT": outputs["apimGatewayUrl"]["value"],
    "AZURE_SEARCH_NAME": outputs["searchServiceName"]["value"],
    "AZURE_SEARCH_RESOURCE_ID": outputs["searchResourceId"]["value"],
    "AZURE_SEARCH_ENDPOINT": outputs["searchEndpoint"]["value"],
}
generated_ts = datetime.now(timezone.utc).isoformat()
env_content = (
    "# BYOM capability testing - generated by 01_deploy_infra.ipynb\n"
    f"# Generated UTC: {generated_ts}\n\n"
    + "\n".join(f"{key}={value}" for key, value in env_values.items())
    + "\n"
)
env_path.write_text(env_content, encoding="utf-8")
print(f"Wrote {env_path}")

## Summary

The resource foundation is provisioned and `env/.env` contains its identifiers/endpoints. BYOM tests and Foundry IQ connections are later work.

When finished, follow the README cleanup instructions: select the saved subscription, run `X_destroy.ipynb`, then purge the saved APIM service with the documented command.